In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Image size should be small for ESP32
IMG_SIZE = 64
NUM_CLASSES = 38

# Load dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    "train",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    label_mode="categorical"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "valid",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    label_mode="categorical"
)

# Normalize images
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

# Tiny CNN Model
model = models.Sequential([
    
    layers.Conv2D(8, (3,3), activation='relu', input_shape=(64,64,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(16, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(64, activation='relu'),

    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.summary()

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)




Found 56666 files belonging to 38 classes.
Found 14163 files belonging to 38 classes.
Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_6 (Conv2D)           (None, 62, 62, 8)         224       
                                                                 
 max_pooling2d_12 (MaxPoolin  (None, 31, 31, 8)        0         
 g2D)                                                            
                                                                 
 conv2d_7 (Conv2D)           (None, 29, 29, 16)        1168      
                                                                 
 max_pooling2d_13 (MaxPoolin  (None, 14, 14, 16)       0         
 g2D)                                                            
                                                                 
 conv2d_8 (Conv2D)           (None, 12, 12, 32)        4640      
                                  

KeyboardInterrupt: 

In [ ]:

# -------------------------------
# Save Keras Model
# -------------------------------
model.save("light_version/plant_disease_model.h5")

# -------------------------------
# Convert to TensorFlow Lite
# -------------------------------
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Enable optimization
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

# Save TFLite model
with open("light_version/plant_disease_model_lite_version.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved successfully!")

INFO:tensorflow:Assets written to: C:\Users\ratha\AppData\Local\Temp\tmpqnm_iu07\assets


INFO:tensorflow:Assets written to: C:\Users\ratha\AppData\Local\Temp\tmpqnm_iu07\assets


TFLite model saved successfully!


: 

In [5]:
def representative_data_gen():
    for images, _ in train_ds.take(100):
        yield [images]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_quant_model = converter.convert()

with open("light_version/plant_disease_tiny_int8.tflite", "wb") as f:
    f.write(tflite_quant_model)

INFO:tensorflow:Assets written to: C:\Users\ratha\AppData\Local\Temp\tmpc14607ob\assets


INFO:tensorflow:Assets written to: C:\Users\ratha\AppData\Local\Temp\tmpc14607ob\assets
c:\Users\ratha\anaconda3\envs\tensorflow_env\lib\site-packages\tensorflow\lite\python\convert.py:766: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn("Statistics for quantized inputs were expected, but not "


In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = 64
NUM_CLASSES = 38

train_ds = tf.keras.utils.image_dataset_from_directory(
    "train",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    label_mode="categorical"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "valid",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    label_mode="categorical"
)

normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

model = models.Sequential([

    layers.Input(shape=(64,64,3)),

    layers.Conv2D(8,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(16,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    # Replace Flatten with this
    layers.GlobalAveragePooling2D(),

    layers.Dense(64,activation='relu'),

    layers.Dense(NUM_CLASSES,activation='softmax')
])

model.summary()

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100
)

Found 70295 files belonging to 38 classes.
Found 17572 files belonging to 38 classes.
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 62, 62, 8)         224       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 31, 31, 8)        0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 29, 29, 16)        1168      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 14, 14, 16)       0         
 2D)                                                             
                                                                 
 conv2d_2 (Conv2D)           (None, 12, 12, 32)        4640      
                                    

In [2]:
# Recording this model history
import json
with open("ESP32_model/training_history.json", 'w') as f:
    json.dump(history.history,f)

In [3]:

# ----------------------------
# Save Keras Model
# ----------------------------
model.save("ESP32_model/plant_disease_model_small.h5")

# ----------------------------
# Representative Dataset for INT8
# ----------------------------
def representative_data_gen():
    for images, _ in train_ds.take(100):
        yield [images]

# ----------------------------
# Convert to TFLite INT8
# ----------------------------
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.representative_dataset = representative_data_gen

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_quant_model = converter.convert()

# ----------------------------
# Save Quantized Model
# ----------------------------
with open("ESP32_model/plant_disease_tiny_int8.tflite", "wb") as f:
    f.write(tflite_quant_model)

print("INT8 Quantized model saved successfully!")

INFO:tensorflow:Assets written to: C:\Users\ratha\AppData\Local\Temp\tmp7mwq2fjl\assets


INFO:tensorflow:Assets written to: C:\Users\ratha\AppData\Local\Temp\tmp7mwq2fjl\assets
c:\Users\ratha\anaconda3\envs\tensorflow_env\lib\site-packages\tensorflow\lite\python\convert.py:766: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn("Statistics for quantized inputs were expected, but not "


INT8 Quantized model saved successfully!


In [ ]:
xxd -i final_light_version/plant_disease_tiny_int8.tflite > model.h